In [ ]:
# Imports

## Core libraries
import numpy as np
import matplotlib.pyplot as plt

## TensorFlow / Keras
import tensorflow as tf
import keras
from keras import ops, layers
from keras.callbacks import Callback
from keras.models import Sequential, Model
from keras.layers import Input, Conv2D, Dense


In [ ]:
# Setting seeds

np.random.seed(1999)
tf.random.set_seed(1999)


In [ ]:
# Loading the dataset

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

## Feature scaling (mapping pixel values to [0,1])
'''
This is used as neural network training via gradient descent is sensitive to the
scale of inputs (through activations a = Wx + b). This causes gradients to vanish
and makes the loss landscape poorly conditioned.
'''
x_train_full = x_train_full / 255.0
x_test = x_test / 255.0

''' Recalling our y values are our categories '''
y_train_full = keras.utils.to_categorical(y_train_full, 10)
y_test = keras.utils.to_categorical(y_test, 10)

## Setting up our validation dataset
n_val = int(0.2 * len(x_train_full))
perm = np.random.permutation(len(x_train_full))
val_idx, train_idx = perm[:n_val], perm[n_val:]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]

## Check
print(f"train: {x_train.shape} val: {x_val.shape} test: {x_test.shape}")


In [ ]:
# Augmenting the dataset
'''
This involves creating new training examples by applying label preserving
transformations to existing examples e.g. flips, crops, scaling, colour jitter,
adding noise, ...
'''

'''
Applying three transformations in sequence, we pad the image with black 0s,
then randomly crop the newly padded image (so the model learns that a shifted
horse picture is still a horse), and then randomly mirroring the image from left
to right.
'''
augment = keras.Sequential([layers.ZeroPadding2D(padding=4),
                            layers.RandomCrop(32, 32),
                            layers.RandomFlip("horizontal")])

batch_size = 128

'''
This builds the full training data pipeline, where:
- The from_tensor_slices((x_train, y_train)) wraps the numpy arrays into a
  tf.data.Dataset that has image label pairs one at a time.
- For .shuffle(len(x_train)) - this shifts the entire dataset after each epoch.
- For .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
  this applies the augmentation pipeline to each batch of images whilst leaving
  labels untouched. The training=True is important because RandomCrop and
  RandomFlip only apply their transformations during training. The
  num_parallel_calls=tf.data.AUTOTUNE lets TensorFlow decide how many batches to
  augment in parallel across CPU threads.
'''
train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(x_train)).batch(batch_size).map(lambda x, y: (augment(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))

### Recalling that there is no augmentation for validation and test datasets
val_ds  = (tf.data.Dataset.from_tensor_slices((x_val,  y_val ))
           .batch(batch_size).prefetch(tf.data.AUTOTUNE))
test_ds = (tf.data.Dataset.from_tensor_slices((x_test, y_test))
           .batch(batch_size).prefetch(tf.data.AUTOTUNE))


In [ ]:
# Residual block (no dropout)
'''
Here is the specification requirements for a single residual block (the repeating
unit in ResNet-20). Instead of learning the mapping between the input and the
output, we instead learn the difference between the input and the desired output
(the residual).

In this, the shortcut path passes the input unchanged (albeit with a channel
count change to match shapes at times). These two paths are added together for
a final ReLU.
'''

### We recall that a convolution slides a small window (a kernel/filter) across our grid.
### A single kernel produces one output grid (one channel). One kernel might learn to detect vertical edges, another colour gradients, and so on.
### We thus use 16 kernels simultaneously to produce an output with 16 channels.

class ResidualBlock(keras.layers.Layer):

    '''
    The init method constructs everything which doesn't fit into the other categories.
    In this, we construct the two convolutional layers and the ReLU activation.
    Everything here is created once and reused when the block processes an input.
    Projection is handled separately, as the input's channel count changes.
    '''
    def __init__(self, filters, stride=1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.stride = stride

        ### The same padding here adds just enough padding at each convolution step so the output spatial dimensions match the input.
        ### The kernel initialiser establishes starting weights for the kernels such that they are not all zero (each kernel learns the same thing), too large (where activations explode), or too small (vanishing to zero).
        self.conv1 = layers.Conv2D(filters, 3, strides=stride, padding="same",
                                   kernel_regularizer=keras.regularizers.L2(1e-4),
                                   kernel_initializer="he_normal")

        ### Batch normalisation ensures each channel has a mean of 0 and a variance of 1.
        ### This is necessary for faster and more stable training as the weights update during training, causing our outputs to shift considerably (and calibrated to the old weights).
        ### The purpose of batch normalisation is to ensure a stable range of outputs.
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv2D(filters, 3, padding="same",
                                   kernel_regularizer=keras.regularizers.L2(1e-4),
                                   kernel_initializer="he_normal")
        self.bn2 = layers.BatchNormalization()
        self.relu = layers.Activation("relu")

    ### Going deeper into the network, we want to detect increasingly abstract features.
    ### We therefore downsample (have a bigger stride) as we detect combinations.
    ### We compensate for that by increasing the number of channels.

    '''
    The build method handles the projection shortcut (and handles shape-matching if needed).
    '''
    def build(self, input_shape):
        in_channels = input_shape[-1]

        if self.stride != 1 or in_channels != self.filters:
            self.proj = layers.Conv2D(self.filters, 1, strides=self.stride, padding="valid",
                                      kernel_regularizer=keras.regularizers.L2(1e-4),
                                      kernel_initializer="he_normal")
            self.bn_proj = layers.BatchNormalization()
        else:
            self.proj = None

        super().build(input_shape)

    '''
    The call method runs the actual forward pass of the model organising the
    split into the main convolutional path and the shortcut path, then summing
    them and applying ReLU.
    '''
    def call(self, x):
        if self.proj is not None:
            shortcut = self.bn_proj(self.proj(x))
        else:
            shortcut = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        return self.relu(out + shortcut)

    '''
    The get_config method enables serialisation so Keras can save/load the model.
    '''
    def get_config(self):
        config = super().get_config()
        config.update({"filters": self.filters,
                       "stride": self.stride})
        return config


In [ ]:
# build_resnet20_vib — VIB + SWAG variant

'''
The feature extractor follows the deterministic ResNet-20 specification above.
The VIB change is after global average pooling: instead of sending the feature
vector directly to the classifier, we map it to the parameters of a diagonal
Gaussian bottleneck, sample z, and classify from z.

SWAG is added later during training by collecting weight snapshots and sampling
from the fitted Gaussian approximation. The architecture itself is therefore
just the VIB-ResNet-20 architecture.
'''

class VIBSampling(keras.layers.Layer):

    """
    Reparameterisation layer for VIB.

    Given mu and log_var, sample z = mu + sigma * epsilon and add the VIB KL
    penalty beta * KL(q(z|x) || N(0, I)) to the model loss.
    """
    def __init__(self, beta=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.beta = beta

    def call(self, inputs):
        mu, log_var = inputs
        eps = tf.random.normal(shape=tf.shape(mu), dtype=mu.dtype)
        z = mu + tf.exp(0.5 * log_var) * eps

        kl_per_example = -0.5 * tf.reduce_sum(
            1.0 + log_var - tf.square(mu) - tf.exp(log_var), axis=-1
        )
        self.add_loss(self.beta * tf.reduce_mean(kl_per_example))

        return z

    def get_config(self):
        config = super().get_config()
        config.update({"beta": self.beta})
        return config


def build_vib_resnet20(beta=1e-3, latent_dim=64):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(16, 3, padding="same",
                      kernel_regularizer=keras.regularizers.L2(1e-4),
                      kernel_initializer="he_normal")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    for _ in range(3):
        x = ResidualBlock(16)(x)

    x = ResidualBlock(32, stride=2)(x)
    for _ in range(2):
        x = ResidualBlock(32)(x)

    x = ResidualBlock(64, stride=2)(x)
    for _ in range(2):
        x = ResidualBlock(64)(x)

    ### Global average pooling takes the block's output and turns it into a flat
    ### feature vector. In the deterministic version this went straight into the
    ### final dense layer; here it becomes the input to the VIB bottleneck.
    x = layers.GlobalAveragePooling2D()(x)

    z_mu = layers.Dense(latent_dim,
                        kernel_initializer="he_normal",
                        name="z_mu")(x)
    z_log_var = layers.Dense(latent_dim,
                             kernel_initializer="zeros",
                             bias_initializer="zeros",
                             name="z_log_var")(x)
    z = VIBSampling(beta=beta, name="vib_bottleneck")([z_mu, z_log_var])

    outputs = layers.Dense(10, kernel_initializer="he_normal")(z)

    return keras.Model(inputs, outputs)


In [ ]:
# Building VIB-SWAG model
'''
β is loaded from the VIB-alone sweep results (vib_alone_results.npz) so that
VIB-alone and VIB-SWAG use identical compression strength. This keeps the
bottleneck uncertainty component directly comparable between the two models —
any difference in bottleneck magnitude reflects the interaction with SWAG,
not a difference in β.

If vib_alone_results.npz does not exist (e.g. running this notebook before the
VIB-alone notebook has been executed), we fall back to β = 1e-3, which is the
value used in the original VIB paper (Alemi et al. 2016) and a reasonable default.
'''
try:
    _vib_data = np.load("vib_alone_results.npz", allow_pickle=True)
    beta = float(_vib_data["beta"])
    print(f"Loaded β = {beta:.0e} from VIB-alone sweep (vib_alone_results.npz)")
except FileNotFoundError:
    beta = 1e-3
    print(f"vib_alone_results.npz not found — using fallback β = {beta:.0e}")

latent_dim = 64  # equal to the ResNet-20 pooled feature width

model = build_vib_resnet20(beta=beta, latent_dim=latent_dim)
model.summary()


In [ ]:
# Initial VIB training

'''
Step-wise learning-rate schedule: drop by 10x at epochs 100 and 150.
This is unchanged from the deterministic notebook so the combined VIB-SWAG
model starts from the same optimisation recipe as the VIB-alone model.
'''
def lr_schedule(epoch, lr):
    if epoch in (100, 150):
        return lr * 0.1
    return lr


In [ ]:
# Compiling initial VIB model

### This doesn't run any training or touch any data, but it configures the optimiser (how weights are updated), the loss function, and optionally any metrics we want to track.
### The VIB KL term is added inside the VIBSampling layer via self.add_loss(...), so the compile loss remains categorical cross-entropy.
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
              loss=keras.losses.CategoricalCrossentropy(from_logits=True),
              metrics=["categorical_accuracy"])


In [ ]:
# Callbacks for initial VIB checkpoint

callbacks = [keras.callbacks.ModelCheckpoint("vib_swag_initial_best.weights.h5",
                                             monitor="val_loss",
                                             save_best_only=True,
                                             save_weights_only=True),
             keras.callbacks.LearningRateScheduler(lr_schedule, verbose=1)] # calls lr_schedule function each epoch to reduce learning rate


In [ ]:
# Initial VIB training run

history = model.fit(train_ds,
                    epochs=200,
                    validation_data=val_ds,
                    callbacks=callbacks)


model.load_weights("vib_swag_initial_best.weights.h5") # Loading best initial VIB checkpoint

# Final evaluation on test set before SWAG collection.
# This is a single stochastic pass, so the double-MC evaluation cell below is
# the better number to report for the final VIB-SWAG predictive distribution.
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nInitial VIB single-sample test loss: {test_loss:.4f}  Initial VIB single-sample test accuracy: {test_acc:.4f}")


In [ ]:
# Plotting initial VIB training curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

## Plot 1
ax1.plot(history.history["loss"], label="train loss")
ax1.plot(history.history["val_loss"], label="val loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
ax1.set_title("Initial VIB ResNet-20 — Loss"); ax1.legend()

## Plot 2
ax2.plot(history.history["categorical_accuracy"], label="train acc")
ax2.plot(history.history["val_categorical_accuracy"], label="val acc")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy")
ax2.set_title("Initial VIB ResNet-20 — Accuracy"); ax2.legend()
plt.tight_layout(); plt.show()


In [ ]:
# SWAG helper functions
'''
SWAG is fit after the initial VIB training run. We keep the architecture fixed,
continue training with a constant learning rate, and collect parameter snapshots.

For stability, only trainable variables are sampled. BatchNorm moving means and
moving variances are left at the values learned during SWAG collection. This is
a small practical deviation from sampling every array in model.get_weights(),
because directly sampling BatchNorm moving variances can create negative values.
'''

def flatten_trainable_variables(model):
    return np.concatenate([v.numpy().ravel() for v in model.trainable_variables]).astype("float32")


def set_trainable_variables_from_vector(model, vector):
    pointer = 0
    for variable in model.trainable_variables:
        shape = variable.shape
        size = int(np.prod(shape))
        new_value = vector[pointer:pointer + size].reshape(shape)
        variable.assign(new_value)
        pointer += size

    if pointer != len(vector):
        raise ValueError("Vector length did not match the model's trainable variables.")


def fit_swag_distribution(snapshots):
    snapshots = np.asarray(snapshots, dtype="float32")
    if snapshots.ndim != 2 or snapshots.shape[0] < 2:
        raise ValueError("Need at least two SWAG snapshots to estimate covariance.")

    mean = snapshots.mean(axis=0)
    diag_var = snapshots.var(axis=0, ddof=1)
    deviations = snapshots - mean

    return {"mean": mean,
            "diag_var": diag_var,
            "deviations": deviations,
            "n_snapshots": snapshots.shape[0]}


def sample_swag_weights(swag, rng=None, diag_scale=1.0, low_rank_scale=1.0):
    if rng is None:
        rng = np.random.default_rng()

    mean = swag["mean"]
    diag_var = swag["diag_var"]
    deviations = swag["deviations"]
    k = swag["n_snapshots"]

    ### SWAG commonly uses a 1/sqrt(2) split between diagonal and low-rank parts.
    diag_sample = np.sqrt(0.5) * diag_scale * np.sqrt(diag_var + 1e-30) * rng.normal(size=mean.shape)
    low_rank_sample = np.sqrt(0.5) * low_rank_scale * deviations.T @ rng.normal(size=k) / np.sqrt(k - 1)

    return (mean + diag_sample + low_rank_sample).astype("float32")


class SWAGSnapshotCallback(keras.callbacks.Callback):

    def __init__(self, collect_every=2):
        super().__init__()
        self.collect_every = collect_every
        self.snapshots = []

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.collect_every == 0:
            self.snapshots.append(flatten_trainable_variables(self.model))
            print(f"SWAG snapshot {len(self.snapshots)} collected at SWAG epoch {epoch + 1}")


In [ ]:
# SWAG collection training
'''
After initial VIB convergence, we switch to a constant learning rate and keep
training with the same VIB loss. Every c epochs, we take a trainable-weight
snapshot. These snapshots define the SWAG mean, diagonal variance and low-rank
covariance directions.
'''

swag_learning_rate = 0.01
swag_epochs = 40
swag_collect_every = 2

swag_callback = SWAGSnapshotCallback(collect_every=swag_collect_every)

model.compile(optimizer=keras.optimizers.SGD(learning_rate=swag_learning_rate, momentum=0.9),
              loss=keras.losses.CategoricalCrossentropy(from_logits=True),
              metrics=["categorical_accuracy"])

swag_history = model.fit(train_ds,
                         epochs=swag_epochs,
                         validation_data=val_ds,
                         callbacks=[swag_callback])

swag = fit_swag_distribution(swag_callback.snapshots)
set_trainable_variables_from_vector(model, swag["mean"])
model.save_weights("vib_swag_mean.weights.h5")

print(f"Collected {swag['n_snapshots']} SWAG snapshots")
print(f"SWAG parameter dimension: {swag['mean'].shape[0]}")


In [ ]:
# Loading SVHN as the out-of-distribution reference set
'''
Per the proposal §3.5, SVHN (street view of house numbers) is used as the
OOD test set. SVHN images are 32x32 colour photos like CIFAR-10, but of
digits, so a CIFAR-10 trained model should ideally report higher predictive
entropy on SVHN inputs than on its in-distribution CIFAR-10 test images.

The whole SVHN test split is ~26k examples; we load it via
tensorflow_datasets and rescale to [0, 1] to match CIFAR-10 preprocessing.
'''
import tensorflow_datasets as tfds

svhn = tfds.load("svhn_cropped", split="test", as_supervised=True)
svhn_images = []
for img, _ in svhn:
    svhn_images.append(img.numpy())
x_svhn = np.stack(svhn_images).astype("float32") / 255.0

print(f"SVHN OOD set: {x_svhn.shape}")


In [ ]:
# Loading CIFAR-10-C as the second out-of-distribution reference set
'''
Per the proposal §3.5, CIFAR-10-C (Hendrycks & Dietterich 2019) is the
second OOD dataset. Unlike SVHN which represents a full domain shift,
CIFAR-10-C tests calibration under in-distribution corruption: blur, noise,
weather effects, digital artefacts, etc. This is a distinct failure mode
from domain shift and is evaluated separately.

We load all 15 standard corruption types at severity 3 (mid-level) and
concatenate them into a single flat evaluation array. Labels are the
original CIFAR-10 test labels (repeated once per corruption type), so we
can also report accuracy under corruption alongside OOD confidence.

TFDS name format: "cifar10_corrupted/{corruption_type}_{severity}"
'''

# 15 standard corruptions from Hendrycks & Dietterich (2019)
CORRUPTIONS = [
    "brightness", "contrast", "defocus_blur", "elastic_transform",
    "fog", "frost", "glass_blur", "gaussian_noise",
    "impulse_noise", "jpeg_compression", "motion_blur",
    "pixelate", "shot_noise", "snow", "zoom_blur"
]
SEVERITY = 3  # severity levels 1–5; 3 is the standard mid-level benchmark

cifar10c_images = []
cifar10c_labels = []
loaded_corruptions = []

for corruption in CORRUPTIONS:
    ds_name = f"cifar10_corrupted/{corruption}_{SEVERITY}"
    try:
        ds = tfds.load(ds_name, split="test", as_supervised=True)
        for img, label in ds:
            cifar10c_images.append(img.numpy())
            cifar10c_labels.append(label.numpy())
        loaded_corruptions.append(corruption)
    except Exception as e:
        print(f"  Warning — skipping {corruption}: {e}")

x_cifar10c = np.stack(cifar10c_images).astype("float32") / 255.0
y_cifar10c = np.array(cifar10c_labels)   # integer labels 0–9

print(f"CIFAR-10-C OOD set: {x_cifar10c.shape}")
print(f"  Severity: {SEVERITY}  |  Corruptions loaded: {len(loaded_corruptions)}/{len(CORRUPTIONS)}")


In [ ]:
# Uncertainty decomposition (three-way, proposal §3.4)
"""
decompose_uncertainty splits the total predictive uncertainty of the VIB-SWAG
double-MC predictive into three additive, per-input components, following §3.4.

Given per-sample probabilities all_probs of shape (S, M, N, C) — S SWAG weight
samples, M VIB bottleneck samples per weight sample — write
    p_bar    = mean over (s, m)            # (N, C)   full predictive
    p_bar_s  = mean over m, for fixed s    # (S, N, C) per-weight predictive
and let H[.] be Shannon entropy (in nats). Then:

    total            = H[p_bar]
    weight_epistemic = H[p_bar] - (1/S) Σ_s H[p_bar_s]
    bottleneck       = (1/S) Σ_s ( H[p_bar_s] - (1/M) Σ_m H[p_{s,m}] )
    residual         = (1/SM) Σ_{s,m} H[p_{s,m}]

weight_epistemic measures the uncertainty removed by fixing the weights;
bottleneck the uncertainty removed by then fixing the representation z;
residual the classifier's remaining uncertainty with weights and z both fixed.
By construction the three sum to total (a telescoping of conditional entropies),
which is asserted below.
"""

def _entropy(p, axis=-1, eps=1e-12):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1.0)
    return -np.sum(p * np.log(p), axis=axis)


def decompose_uncertainty(all_probs):
    """
    Parameters
    ----------
    all_probs : np.ndarray, shape (S, M, N, C)
        Per-sample softmax probabilities from the double-MC procedure.

    Returns
    -------
    dict with keys 'total', 'weight_epistemic', 'bottleneck', 'residual',
    each a (N,) array of per-input predictive uncertainty in nats.
    """
    ap = np.asarray(all_probs)
    if ap.ndim != 4:
        raise ValueError(f"Expected all_probs of shape (S, M, N, C); got {ap.shape}")

    mean_p   = ap.mean(axis=(0, 1))          # (N, C)   p_bar
    mean_p_s = ap.mean(axis=1)               # (S, N, C) p_bar_s

    H_total = _entropy(mean_p)               # (N,)     H[p_bar]
    H_s     = _entropy(mean_p_s, axis=-1)    # (S, N)   H[p_bar_s]
    H_sm    = _entropy(ap, axis=-1)          # (S, M, N) H[p_{s,m}]

    weight_epistemic = H_total - H_s.mean(axis=0)
    bottleneck       = (H_s - H_sm.mean(axis=1)).mean(axis=0)
    residual         = H_sm.mean(axis=(0, 1))

    # Additivity sanity check: components must reconstruct the total entropy.
    recon = weight_epistemic + bottleneck + residual
    assert np.allclose(recon, H_total, atol=1e-5, rtol=1e-4), (
        "Three-way decomposition does not sum to total "
        f"(max abs error {np.abs(recon - H_total).max():.2e})")

    return {"total": H_total,
            "weight_epistemic": weight_epistemic,
            "bottleneck": bottleneck,
            "residual": residual}


In [ ]:
# VIB-SWAG double-MC predictions — joint inference with BN recalibration
'''
Two fixes applied here relative to the original design:

1. BN recalibration per weight sample (same fix as the SWAG baseline).
   set_trainable_variables_from_vector only loads trainable weights; the BN
   layers' non-trainable running statistics (moving_mean, moving_variance)
   stay at their SWA-mean values. Each θ_s shifts the feature distributions,
   so those frozen stats are mismatched. Fix: one forward pass through
   train_ds with training=True after each weight load. The VIBSampling layer
   fires unconditionally (no training-flag guard), so it also samples z
   during this pass — but the outputs are discarded; only the side-effect on
   BN running stats is wanted.

2. Joint loop over all evaluation sets.
   All three datasets share the same S weight samples (same rng seed),
   so BN recalibration is paid once per weight sample rather than once per
   dataset × per sample. Sharing weight samples also makes the per-dataset
   uncertainty estimates directly comparable: any difference reflects the
   data, not sampling noise in θ.

3. Save / restore non_trainable_variables.
   After sampling, both trainable weights AND BN running stats are restored
   to the SWA mean state so downstream cells see the model in a known state.

Note on all_probs_cifar10c memory: shape (S, M, N_c10c, 10) = (30, 10, 150k, 10)
≈ 1.8 GB. It is computed here but not saved to disk by default (the pre-computed
decomposition arrays are saved instead). Uncomment the all_probs_cifar10c key
in the save cell if you need to recompute decompositions offline.
'''

n_weight_samples = 30   # S: SWAG weight samples
n_rep_samples    = 10   # M: VIB bottleneck samples per weight sample
S, M = n_weight_samples, n_rep_samples

# Save full model state: trainable weights AND non-trainable BN running stats
original_weights       = [w.numpy().copy() for w in model.trainable_weights]
original_non_trainable = [v.numpy().copy() for v in model.non_trainable_variables]


def predict_vib_swag(eval_datasets, swag, model, S=S, M=M,
                     batch_size=128, seed=1999, bn_calib_batches=None):
    '''
    Parameters
    ----------
    eval_datasets : dict of {name: data}
        data is a tf.data.Dataset (yielding (x, y) pairs) or a numpy array.
    swag : dict
        Output of fit_swag_distribution.
    model : keras.Model
        The VIB-ResNet-20 model (weights will be temporarily modified).
    S : int
        Number of SWAG weight samples.
    M : int
        Number of VIB bottleneck samples per weight sample.
    bn_calib_batches : int or None
        Training batches for BN recalibration. None = full train_ds.

    Returns
    -------
    dict of {name: {all_probs, mean_probs, uncertainty}}
        all_probs  : (S, M, N, 10)
        mean_probs : (N, 10)
        uncertainty: dict with keys total, weight_epistemic, bottleneck, residual
    '''
    rng   = np.random.default_rng(seed)
    bn_ds = train_ds if bn_calib_batches is None else train_ds.take(bn_calib_batches)

    # Build inference dataset wrappers
    inf_ds = {}
    for name, data in eval_datasets.items():
        if isinstance(data, np.ndarray):
            ds_inf = tf.data.Dataset.from_tensor_slices(data).batch(batch_size)
            get_x  = lambda b: b          # plain tensor batches
        else:
            ds_inf = data
            get_x  = lambda b: b[0]       # (x, y) pairs — extract x
        inf_ds[name] = (ds_inf, get_x)

    # weight_draws[name]: list of (M, N, 10) arrays, one per weight sample s
    weight_draws = {name: [] for name in eval_datasets}

    for s in range(S):
        # 1. Sample θ_s and load into model (trainable weights only)
        sampled_weights = sample_swag_weights(swag, rng=rng)
        set_trainable_variables_from_vector(model, sampled_weights)

        # 2. BN recalibration: update moving_mean / moving_variance for θ_s.
        #    VIBSampling also fires here (unconditionally) but outputs are discarded.
        for bn_batch in bn_ds:
            model(bn_batch[0], training=True)

        # 3. M bottleneck samples per dataset.
        #    training=False uses the calibrated BN running stats.
        #    VIBSampling draws a fresh z for every call, giving M independent
        #    representation samples from q(z | x, θ_s).
        for name, (ds_inf, get_x) in inf_ds.items():
            rep_probs = []
            for m in range(M):
                per_batch = []
                for batch in ds_inf:
                    logits = model(get_x(batch), training=False)
                    per_batch.append(tf.nn.softmax(logits, axis=-1).numpy())
                rep_probs.append(np.concatenate(per_batch, axis=0))   # (N, 10)
            weight_draws[name].append(np.stack(rep_probs, axis=0))    # (M, N, 10)

        print(f"  Weight sample {s + 1:02d}/{S} complete")

    # Restore full model state
    for w, orig in zip(model.trainable_weights, original_weights):
        w.assign(orig)
    for v, orig in zip(model.non_trainable_variables, original_non_trainable):
        v.assign(orig)

    # Assemble (S, M, N, 10) arrays and run decomposition
    results = {}
    for name in eval_datasets:
        all_probs   = np.stack(weight_draws[name], axis=0)  # (S, M, N, 10)
        mean_probs  = all_probs.mean(axis=(0, 1))            # (N, 10)
        uncertainty = decompose_uncertainty(all_probs)
        results[name] = {"all_probs":   all_probs,
                          "mean_probs":  mean_probs,
                          "uncertainty": uncertainty}
    return results


# Joint inference over all three evaluation sets
print(f"Running VIB-SWAG double-MC inference "
      f"(S={S}, M={M}, BN recalibration: full train_ds)...")

results = predict_vib_swag(
    eval_datasets={"test": test_ds, "svhn": x_svhn, "cifar10c": x_cifar10c},
    swag=swag, model=model, S=S, M=M, seed=1999
)

# Unpack for downstream cells and npz save
all_probs_test,     probs_test     = results["test"]["all_probs"],     results["test"]["mean_probs"]
all_probs_svhn,     probs_svhn     = results["svhn"]["all_probs"],     results["svhn"]["mean_probs"]
all_probs_cifar10c, probs_cifar10c = results["cifar10c"]["all_probs"], results["cifar10c"]["mean_probs"]

uncertainty_test     = results["test"]["uncertainty"]
uncertainty_svhn     = results["svhn"]["uncertainty"]
uncertainty_cifar10c = results["cifar10c"]["uncertainty"]

preds_test     = np.argmax(probs_test, axis=1)
labels_test    = np.argmax(y_test, axis=1)
preds_cifar10c = np.argmax(probs_cifar10c, axis=1)

print(f"\nVIB-SWAG double-MC test accuracy:          {(preds_test == labels_test).mean():.4f}")
print(f"Accuracy under corruption (severity {SEVERITY}): {(preds_cifar10c == y_cifar10c).mean():.4f}")

print(f"\nall_probs_test:     {all_probs_test.shape}   (S, M, N_test, C)")
print(f"all_probs_svhn:     {all_probs_svhn.shape}   (S, M, N_svhn, C)")
print(f"all_probs_cifar10c: {all_probs_cifar10c.shape}   (S, M, N_c10c, C)")

print(f"\nTest set uncertainty breakdown (mean over N_test):")
print(f"  total:            {uncertainty_test['total'].mean():.4f}")
print(f"  weight_epistemic: {uncertainty_test['weight_epistemic'].mean():.4f}")
print(f"  bottleneck:       {uncertainty_test['bottleneck'].mean():.4f}")
print(f"  residual:         {uncertainty_test['residual'].mean():.4f}")
print(f"  (sum check):      {(uncertainty_test['weight_epistemic'] + uncertainty_test['bottleneck'] + uncertainty_test['residual']).mean():.4f}  (should ≈ total)")


In [ ]:
# Saving raw outputs for the evaluation file
'''
Save format (shared across all benchmarks where applicable).

The key difference from the other baselines is the 4D all_probs shape (S, M, N, C)
and the pre-computed three-way decomposition arrays. The eval notebook reads
the decomposition arrays directly — it does not need to recompute them from
all_probs.

Memory note: all_probs_cifar10c has shape (30, 10, 150k, 10) ≈ 1.8 GB and is
NOT saved by default. The pre-computed decomposition arrays for CIFAR-10-C are
saved instead (total_cifar10c etc.) and are all the eval notebook needs.
Uncomment the all_probs_cifar10c line below if you need the raw arrays for
offline analysis.

Keys:
  model_name                - identifier
  beta                      - VIB compression strength (from VIB-alone sweep)
  latent_dim                - bottleneck dimensionality
  n_weight_samples          - S (SWAG samples)
  n_rep_samples             - M (bottleneck samples per weight sample)
  probs_test                - (N_test, 10)      mean predictive dist., CIFAR-10
  labels_test               - (N_test,)         true labels (integer 0–9)
  preds_test                - (N_test,)         argmax of probs_test
  probs_svhn                - (N_svhn, 10)      mean predictive dist., SVHN
  probs_cifar10c            - (N_c10c, 10)      mean predictive dist., CIFAR-10-C
  labels_cifar10c           - (N_c10c,)         true labels, CIFAR-10-C
  preds_cifar10c            - (N_c10c,)         argmax of probs_cifar10c
  cifar10c_severity         - severity level used (3)
  all_probs_test            - (S, M, N_test, 10) full 4D array
  all_probs_svhn            - (S, M, N_svhn, 10) full 4D array
  total_*/weight_epistemic_*/bottleneck_*/residual_*
                            - per-input uncertainty components for test, svhn,
                              cifar10c (each shape (N,))
'''
np.savez("vib_swag_results.npz",
         model_name="vib_swag",
         beta=beta,
         latent_dim=latent_dim,
         swag_learning_rate=swag_learning_rate,
         swag_epochs=swag_epochs,
         swag_collect_every=swag_collect_every,
         n_swag_snapshots=swag["n_snapshots"],
         n_weight_samples=n_weight_samples,
         n_rep_samples=n_rep_samples,
         # In-distribution
         probs_test=probs_test,
         labels_test=labels_test,
         preds_test=preds_test,
         # OOD — full domain shift
         probs_svhn=probs_svhn,
         all_probs_svhn=all_probs_svhn,
         # OOD — in-distribution corruption
         probs_cifar10c=probs_cifar10c,
         labels_cifar10c=y_cifar10c,
         preds_cifar10c=preds_cifar10c,
         cifar10c_severity=SEVERITY,
         # Full 4D arrays (needed for offline recomputation of decomposition)
         all_probs_test=all_probs_test,
         # all_probs_cifar10c=all_probs_cifar10c,  # ≈1.8 GB — uncomment if needed
         # Pre-computed three-way decomposition — test set
         total_test=uncertainty_test["total"],
         weight_epistemic_test=uncertainty_test["weight_epistemic"],
         bottleneck_test=uncertainty_test["bottleneck"],
         residual_test=uncertainty_test["residual"],
         # Pre-computed three-way decomposition — SVHN
         total_svhn=uncertainty_svhn["total"],
         weight_epistemic_svhn=uncertainty_svhn["weight_epistemic"],
         bottleneck_svhn=uncertainty_svhn["bottleneck"],
         residual_svhn=uncertainty_svhn["residual"],
         # Pre-computed three-way decomposition — CIFAR-10-C
         total_cifar10c=uncertainty_cifar10c["total"],
         weight_epistemic_cifar10c=uncertainty_cifar10c["weight_epistemic"],
         bottleneck_cifar10c=uncertainty_cifar10c["bottleneck"],
         residual_cifar10c=uncertainty_cifar10c["residual"])

np.savez("vib_swag_distribution.npz",
         mean=swag["mean"],
         diag_var=swag["diag_var"],
         deviations=swag["deviations"],
         n_snapshots=swag["n_snapshots"])

print("Saved vib_swag_results.npz")
print("Saved vib_swag_distribution.npz")
